# Generate paper figures from existing V2 results

This notebook only reads existing final_results.csv and histories.csv; it does not retrain any model.

The V2 protocol selects the best checkpoint by validation loss and evaluates the official test set once. Therefore, the training curves below use validation loss, not per-epoch test loss.


In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Existing experiment output root on the Windows machine.
RUN_ROOT = Path(r"D:\2025暑期科研\UCRArchive_2018\TwoTower_Project_V2\final_runs_v2")
SEED = 42

OUTPUT_DIR = RUN_ROOT / "paper_figures_v2" / f"seed_{SEED}"
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MAIN_MODELS = ["twotower", "informer", "fedformer", "ts2vec"]
ABLATION_MODELS = ["twotower", "correlation", "transformer_only"]
ALL_MODELS = list(dict.fromkeys(MAIN_MODELS + ABLATION_MODELS))

# Explicitly select the versions confirmed from the final training notebooks.
MODEL_CANDIDATES = {
    "twotower": ["twotower"],
    "transformer_only": ["transformer_only"],
    # Final comparison uses train_twotower_correlation_v2_aligned(3).ipynb.
    "correlation": ["correlation_aligned_v2"],
    # Final tuned lightweight Informer notebook uses MODEL_NAME=informer_probsparse_lightweight.
    "informer": ["informer_probsparse_lightweight"],
    "fedformer": ["fedformer"],
    "ts2vec": ["ts2vec"],
}

MODEL_LABEL = {
    "twotower": "Two-Tower",
    "informer": "Informer",
    "fedformer": "FEDformer",
    "ts2vec": "TS2Vec",
    "correlation": "Correlation Two-Tower",
    "transformer_only": "Transformer-only",
}

MODEL_COLOR = {
    "twotower": "#1f77b4",
    "informer": "#2ca02c",
    "fedformer": "#ff7f0e",
    "ts2vec": "#d62728",
    "correlation": "#ff7f0e",
    "transformer_only": "#7f7f7f",
}

REPRESENTATIVE_DATASETS = ["ArrowHead", "Beef", "Coffee", "ECG200", "ECG5000", "GunPoint"]

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 24,
    "font.weight": "normal",
    "axes.titlesize": 32,
    "axes.titleweight": "normal",
    "axes.labelsize": 32,
    "axes.labelweight": "normal",
    "legend.fontsize": 24,
    "xtick.labelsize": 26,
    "ytick.labelsize": 26,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.linewidth": 1.5,
    "xtick.major.width": 1.5,
    "ytick.major.width": 1.5,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "figure.titlesize": 32,
    "figure.titleweight": "normal",
})

print(f"RUN_ROOT: {RUN_ROOT}")
print(f"FIGURE_DIR: {FIGURE_DIR}")


In [ ]:
def resolve_model_dir(model_key):
    seed_dir = f"seed_{SEED}"
    for candidate in MODEL_CANDIDATES[model_key]:
        candidate_dir = RUN_ROOT / candidate / seed_dir
        if (candidate_dir / "final_results.csv").exists() or (candidate_dir / "histories.csv").exists():
            return candidate_dir
    return None


MODEL_DIRS = {key: resolve_model_dir(key) for key in ALL_MODELS}
presence_rows = []
for key in ALL_MODELS:
    path = MODEL_DIRS[key]
    presence_rows.append({
        "model": key,
        "label": MODEL_LABEL[key],
        "resolved_dir": str(path) if path else "",
        "final_results_exists": bool(path and (path / "final_results.csv").exists()),
        "histories_exists": bool(path and (path / "histories.csv").exists()),
    })

presence = pd.DataFrame(presence_rows)
display(presence)
presence.to_csv(OUTPUT_DIR / "model_presence.csv", index=False)


def require_models(model_keys, filename):
    missing = []
    for key in model_keys:
        path = MODEL_DIRS.get(key)
        if path is None or not (path / filename).exists():
            missing.append(key)
    if missing:
        raise FileNotFoundError(
            f"Missing {filename} for {missing}. Check MODEL_CANDIDATES and RUN_ROOT."
        )


def _filter_completed(df):
    if "status" in df.columns:
        status = df["status"].astype(str).str.lower().str.strip()
        completed = status.isin({"completed", "complete", "success", "ok"})
        if completed.any():
            df = df.loc[completed].copy()
    return df


def load_final_results(model_key):
    path = MODEL_DIRS[model_key] / "final_results.csv"
    df = _filter_completed(pd.read_csv(path))
    required = {"dataset", "test_auc", "test_acc", "test_loss"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")
    df = df.copy()
    df["dataset"] = df["dataset"].astype(str)
    for column in ["test_auc", "test_acc", "test_loss"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")
    return df.drop_duplicates(subset=["dataset"], keep="last")


def load_histories(model_key):
    path = MODEL_DIRS[model_key] / "histories.csv"
    df = pd.read_csv(path)
    required = {"dataset", "epoch", "val_loss"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")
    df = df.copy()
    df["dataset"] = df["dataset"].astype(str)
    df["epoch"] = pd.to_numeric(df["epoch"], errors="coerce")
    df["val_loss"] = pd.to_numeric(df["val_loss"], errors="coerce")
    if "train_loss" in df.columns:
        df["train_loss"] = pd.to_numeric(df["train_loss"], errors="coerce")
    return df.dropna(subset=["dataset", "epoch", "val_loss"])


FINAL = {}
HISTORIES = {}
for key in ALL_MODELS:
    if MODEL_DIRS[key] is None:
        continue
    if (MODEL_DIRS[key] / "final_results.csv").exists():
        FINAL[key] = load_final_results(key)
    if (MODEL_DIRS[key] / "histories.csv").exists():
        HISTORIES[key] = load_histories(key)

print("Loaded final result files:", {k: len(v) for k, v in FINAL.items()})
print("Loaded history files:", {k: len(v) for k, v in HISTORIES.items()})


## Figure 1: overall main-model comparison

This uses the common completed dataset set across the four main models. The values are simple means over datasets, so the comparison is not dominated by the largest UCR datasets.


In [ ]:
def common_datasets(data_map):
    if not data_map:
        return []
    sets = [set(df["dataset"].dropna()) for df in data_map.values()]
    return sorted(set.intersection(*sets))


def aggregate_metrics(data_map, datasets):
    rows = []
    for key, df in data_map.items():
        part = df[df["dataset"].isin(datasets)]
        rows.append({
            "model": key,
            "label": MODEL_LABEL[key],
            "datasets": len(part),
            "AUC": part["test_auc"].mean(),
            "ACC": part["test_acc"].mean(),
            "LOSS": part["test_loss"].mean(),
        })
    return pd.DataFrame(rows)


require_models(MAIN_MODELS, "final_results.csv")
main_final = {key: FINAL[key] for key in MAIN_MODELS}
main_common = common_datasets(main_final)
main_aggregate = aggregate_metrics(main_final, main_common)
display(main_aggregate)
pd.DataFrame({"dataset": main_common}).to_csv(OUTPUT_DIR / "main_common_datasets.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(22, 8), constrained_layout=True)
for ax, metric in zip(axes, ["AUC", "ACC", "LOSS"]):
    bars = ax.bar(
        main_aggregate["label"],
        main_aggregate[metric],
        color=[MODEL_COLOR[key] for key in main_aggregate["model"]],
        alpha=0.9,
    )
    ax.set_title(metric)
    ax.set_ylabel("Mean across datasets")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.25)
    for bar in bars:
        height = bar.get_height()
        ax.annotate(
            f"{height:.3f}",
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=16,
        )
fig.suptitle(f"Main-model comparison on {len(main_common)} common UCR datasets")
fig.savefig(FIGURE_DIR / "figure1_main_model_comparison.png", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "figure1_main_model_comparison.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)


## Figure 2: mean validation-loss curves

This is the V2 replacement for a per-epoch test-loss plot. Each curve is averaged over the common completed datasets.


In [ ]:
require_models(MAIN_MODELS, "histories.csv")
main_histories = {key: HISTORIES[key] for key in MAIN_MODELS}
history_common = common_datasets(main_histories)
pd.DataFrame({"dataset": history_common}).to_csv(OUTPUT_DIR / "history_common_datasets.csv", index=False)

fig, ax = plt.subplots(figsize=(14, 8), constrained_layout=True)
for key in MAIN_MODELS:
    part = main_histories[key][main_histories[key]["dataset"].isin(history_common)]
    curve = part.groupby("epoch", as_index=False)["val_loss"].mean().sort_values("epoch")
    ax.plot(
        curve["epoch"],
        curve["val_loss"],
        linewidth=2.0,
        label=MODEL_LABEL[key],
        color=MODEL_COLOR[key],
    )
ax.set_title(f"Mean validation loss across {len(history_common)} common UCR datasets")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation loss")
ax.legend(frameon=True, facecolor="white", framealpha=0.72, edgecolor="0.75", fancybox=True)
ax.grid(alpha=0.4, linewidth=0.8)
fig.savefig(FIGURE_DIR / "figure2_mean_validation_loss.png", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "figure2_mean_validation_loss.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Figure 2 used {len(history_common)} common datasets.")


## Figure 3: representative-dataset validation-loss curves

The six datasets follow the order used in the original paper draft. Missing datasets are skipped and reported rather than silently treated as zero.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 13), sharey=False, constrained_layout=True)
missing_representative = []
for ax, dataset in zip(axes.flat, REPRESENTATIVE_DATASETS):
    plotted = False
    for key in MAIN_MODELS:
        part = main_histories[key]
        part = part[part["dataset"].eq(dataset)].sort_values("epoch")
        if part.empty:
            missing_representative.append((dataset, key))
            continue
        ax.plot(
            part["epoch"],
            part["val_loss"],
            linewidth=1.8,
            label=MODEL_LABEL[key],
            color=MODEL_COLOR[key],
        )
        plotted = True
    ax.set_title(dataset)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation loss")
    ax.grid(alpha=0.4, linewidth=0.8)
    if not plotted:
        ax.text(0.5, 0.5, "No history", ha="center", va="center", transform=ax.transAxes)

handles, labels = axes.flat[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="upper center", ncol=4, frameon=True, facecolor="white", framealpha=0.72, edgecolor="0.75", fancybox=True, bbox_to_anchor=(0.5, 1.03))
fig.suptitle("Validation-loss curves on representative UCR datasets", y=1.08)
fig.savefig(FIGURE_DIR / "figure3_representative_validation_loss.png", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "figure3_representative_validation_loss.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)
if missing_representative:
    print("Missing representative histories:", missing_representative)
else:
    print("All representative histories were found.")


## Figure 4: ablation comparison

This compares the full Two-Tower model with the correlation and Transformer-only ablations on their common completed datasets.


In [ ]:
require_models(ABLATION_MODELS, "final_results.csv")
ablation_final = {key: FINAL[key] for key in ABLATION_MODELS}
ablation_common = common_datasets(ablation_final)
ablation_aggregate = aggregate_metrics(ablation_final, ablation_common)
display(ablation_aggregate)
pd.DataFrame({"dataset": ablation_common}).to_csv(OUTPUT_DIR / "ablation_common_datasets.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(20, 8), constrained_layout=True)
for ax, metric in zip(axes, ["AUC", "ACC", "LOSS"]):
    bars = ax.bar(
        ablation_aggregate["label"],
        ablation_aggregate[metric],
        color=[MODEL_COLOR[key] for key in ablation_aggregate["model"]],
        alpha=0.9,
    )
    ax.set_title(metric)
    ax.set_ylabel("Mean across datasets")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.25)
    for bar in bars:
        height = bar.get_height()
        ax.annotate(
            f"{height:.3f}",
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=16,
        )
fig.suptitle(f"Ablation comparison on {len(ablation_common)} common UCR datasets")
fig.savefig(FIGURE_DIR / "figure4_ablation_comparison.png", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "figure4_ablation_comparison.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)


## Figure 5: paired improvement over ablations

Positive values favor the full Two-Tower model: higher AUC/ACC and lower loss are better.


In [ ]:
full = FINAL["twotower"].set_index("dataset")
paired_rows = []
for baseline_key in ["correlation", "transformer_only"]:
    baseline = FINAL[baseline_key].set_index("dataset")
    datasets = sorted(set(full.index) & set(baseline.index))
    paired_rows.append({
        "baseline": MODEL_LABEL[baseline_key],
        "AUC improvement": (full.loc[datasets, "test_auc"] - baseline.loc[datasets, "test_auc"]).mean(),
        "ACC improvement": (full.loc[datasets, "test_acc"] - baseline.loc[datasets, "test_acc"]).mean(),
        "LOSS improvement": (baseline.loc[datasets, "test_loss"] - full.loc[datasets, "test_loss"]).mean(),
        "datasets": len(datasets),
    })
paired = pd.DataFrame(paired_rows)
display(paired)

fig, axes = plt.subplots(1, 3, figsize=(20, 8), constrained_layout=True)
for ax, metric in zip(axes, ["AUC improvement", "ACC improvement", "LOSS improvement"]):
    bars = ax.bar(
        paired["baseline"],
        paired[metric],
        color=[MODEL_COLOR["correlation"], MODEL_COLOR["transformer_only"]],
        alpha=0.9,
    )
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(metric.replace(" improvement", ""))
    ax.set_ylabel("Full model minus baseline")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.25)
    for bar in bars:
        height = bar.get_height()
        offset = 3 if height >= 0 else -12
        ax.annotate(
            f"{height:+.3f}",
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, offset),
            textcoords="offset points",
            ha="center",
            va="bottom" if height >= 0 else "top",
            fontsize=16,
        )
fig.suptitle("Paired mean improvement of full Two-Tower over ablations")
fig.savefig(FIGURE_DIR / "figure5_paired_ablation_improvement.png", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "figure5_paired_ablation_improvement.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)
paired.to_csv(OUTPUT_DIR / "paired_ablation_improvement.csv", index=False)


In [ ]:
manifest = {
    "run_root": str(RUN_ROOT),
    "seed": SEED,
    "figure_dir": str(FIGURE_DIR),
    "main_common_dataset_count": len(main_common),
    "history_common_dataset_count": len(history_common),
    "ablation_common_dataset_count": len(ablation_common),
    "representative_datasets": REPRESENTATIVE_DATASETS,
    "model_dirs": {key: str(value) if value else None for key, value in MODEL_DIRS.items()},
    "protocol_note": "V2 selects the best checkpoint by validation loss and evaluates the official test set once; figures use validation curves.",
    "generated_files": sorted(str(path) for path in FIGURE_DIR.glob("*")),
}
(OUTPUT_DIR / "figure_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nGenerated files:")
for path in sorted(FIGURE_DIR.glob("*")):
    print(" -", path.name)
print("\nManifest:", OUTPUT_DIR / "figure_manifest.json")


## Table reference A: main-model summary

The next cells generate reference tables from the same V2 result files. They do not change the experiment results and do not generate final LaTeX yet.


In [ ]:
def with_sample_count(df):
    df = df.copy()
    if "n_samples" in df.columns:
        df["n_samples"] = pd.to_numeric(df["n_samples"], errors="coerce")
    else:
        candidate = next(
            (column for column in ["test_n", "n", "num_samples"] if column in df.columns),
            None,
        )
        if candidate is not None:
            df["n_samples"] = pd.to_numeric(df[candidate], errors="coerce")
        else:
            df["n_samples"] = 1.0
    return df


for key in FINAL:
    FINAL[key] = with_sample_count(FINAL[key])


def weighted_mean(values, weights):
    valid = values.notna() & weights.notna() & (weights > 0)
    if not valid.any():
        return np.nan
    return float(np.average(values[valid], weights=weights[valid]))


def summary_table(model_map, datasets):
    rows = []
    for key, df in model_map.items():
        part = df[df["dataset"].isin(datasets)].copy()
        rows.append({
            "Model": MODEL_LABEL[key],
            "Datasets": len(part),
            "Simple mean AUC": part["test_auc"].mean(),
            "Simple mean ACC": part["test_acc"].mean(),
            "Simple mean LOSS": part["test_loss"].mean(),
            "Weighted AUC": weighted_mean(part["test_auc"], part["n_samples"]),
            "Weighted ACC": weighted_mean(part["test_acc"], part["n_samples"]),
            "Weighted LOSS": weighted_mean(part["test_loss"], part["n_samples"]),
        })
    return pd.DataFrame(rows)


main_summary = summary_table(main_final, main_common)
display(main_summary.round(4))
main_summary.to_csv(OUTPUT_DIR / "table_reference_main_summary.csv", index=False, float_format="%.6f")


## Table reference B: per-dataset metrics for the main models

The wide table has one row per common dataset and separate AUC, ACC, LOSS, and sample-count columns for each model. The long table is easier to filter or use when preparing LaTeX.


In [ ]:
def make_long_table(model_map, datasets):
    rows = []
    for key, df in model_map.items():
        part = df[df["dataset"].isin(datasets)].copy()
        part = part.rename(columns={
            "dataset": "Dataset",
            "test_auc": "AUC",
            "test_acc": "ACC",
            "test_loss": "LOSS",
        })
        part["Model"] = MODEL_LABEL[key]
        rows.append(part[["Dataset", "Model", "AUC", "ACC", "LOSS", "n_samples"]])
    return pd.concat(rows, ignore_index=True).sort_values(["Dataset", "Model"])


def make_wide_table(long_table):
    pieces = []
    for model_label in long_table["Model"].drop_duplicates():
        part = long_table[long_table["Model"].eq(model_label)].copy()
        part = part.drop(columns=["Model"]).set_index("Dataset")
        part = part.rename(columns={
            "AUC": f"{model_label} AUC",
            "ACC": f"{model_label} ACC",
            "LOSS": f"{model_label} LOSS",
            "n_samples": f"{model_label} N",
        })
        pieces.append(part)
    return pd.concat(pieces, axis=1).reset_index().sort_values("Dataset")


main_long = make_long_table(main_final, main_common)
main_wide = make_wide_table(main_long)
display(main_wide.head(20).round(4))
main_long.to_csv(OUTPUT_DIR / "table_reference_main_per_dataset_long.csv", index=False, float_format="%.6f")
main_wide.to_csv(OUTPUT_DIR / "table_reference_main_per_dataset_wide.csv", index=False, float_format="%.6f")
print(f"Main per-dataset table rows: {len(main_wide)}")


## Table reference C: ablation summary and per-dataset metrics


In [ ]:
ablation_summary = summary_table(ablation_final, ablation_common)
display(ablation_summary.round(4))
ablation_summary.to_csv(OUTPUT_DIR / "table_reference_ablation_summary.csv", index=False, float_format="%.6f")

ablation_long = make_long_table(ablation_final, ablation_common)
ablation_wide = make_wide_table(ablation_long)
display(ablation_wide.head(20).round(4))
ablation_long.to_csv(OUTPUT_DIR / "table_reference_ablation_per_dataset_long.csv", index=False, float_format="%.6f")
ablation_wide.to_csv(OUTPUT_DIR / "table_reference_ablation_per_dataset_wide.csv", index=False, float_format="%.6f")
print(f"Ablation per-dataset table rows: {len(ablation_wide)}")


## Table reference D: number of dataset-level wins

This is optional but useful when writing the comparison paragraph. A win means the largest AUC/ACC or the smallest LOSS on that common dataset.


In [ ]:
def winner_counts(long_table):
    rows = []
    for metric, direction in [("AUC", "max"), ("ACC", "max"), ("LOSS", "min")]:
        usable = long_table.dropna(subset=[metric])
        winners = (
            usable.sort_values(metric, ascending=(direction == "min"))
            .groupby("Dataset", as_index=False)
            .first()["Model"]
            .value_counts()
        )
        for model_label in long_table["Model"].drop_duplicates():
            rows.append({
                "Metric": metric,
                "Model": model_label,
                "Dataset wins": int(winners.get(model_label, 0)),
            })
    return pd.DataFrame(rows)


main_wins = winner_counts(main_long)
ablation_wins = winner_counts(ablation_long)
display(main_wins)
display(ablation_wins)
main_wins.to_csv(OUTPUT_DIR / "table_reference_main_dataset_wins.csv", index=False)
ablation_wins.to_csv(OUTPUT_DIR / "table_reference_ablation_dataset_wins.csv", index=False)


In [ ]:
table_manifest = {
    "main_summary": str(OUTPUT_DIR / "table_reference_main_summary.csv"),
    "main_per_dataset_long": str(OUTPUT_DIR / "table_reference_main_per_dataset_long.csv"),
    "main_per_dataset_wide": str(OUTPUT_DIR / "table_reference_main_per_dataset_wide.csv"),
    "ablation_summary": str(OUTPUT_DIR / "table_reference_ablation_summary.csv"),
    "ablation_per_dataset_long": str(OUTPUT_DIR / "table_reference_ablation_per_dataset_long.csv"),
    "ablation_per_dataset_wide": str(OUTPUT_DIR / "table_reference_ablation_per_dataset_wide.csv"),
    "main_dataset_wins": str(OUTPUT_DIR / "table_reference_main_dataset_wins.csv"),
    "ablation_dataset_wins": str(OUTPUT_DIR / "table_reference_ablation_dataset_wins.csv"),
    "note": "Reference tables generated from V2 final_results.csv. No LaTeX formatting applied yet.",
}
(OUTPUT_DIR / "table_reference_manifest.json").write_text(
    json.dumps(table_manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Table reference files were saved under:", OUTPUT_DIR)


## Figure 6: selected favorable cases for Two-Tower

This optional figure selects datasets using an explicit rule: Two-Tower must have higher test AUC, higher test ACC, and lower test LOSS than each of Informer, FEDformer, and TS2Vec. The six strongest qualifying datasets are selected by the average rank of the three positive margins.

This is a descriptive favorable-case figure, not evidence that Two-Tower is best on every UCR dataset. The selection table is saved together with the figure for transparency.


In [ ]:
FAVORABLE_BASELINES = ["informer", "fedformer", "ts2vec"]
FAVORABLE_MAX_DATASETS = 6
DOMINANCE_EPS = 1e-12

require_models(["twotower"] + FAVORABLE_BASELINES, "final_results.csv")
favorable_maps = {key: FINAL[key].set_index("dataset") for key in ["twotower"] + FAVORABLE_BASELINES}
favorable_common = sorted(
    set.intersection(*(set(frame.index) for frame in favorable_maps.values()))
)

dominance_rows = []
for dataset in favorable_common:
    full_row = favorable_maps["twotower"].loc[dataset]
    baseline_rows = [favorable_maps[key].loc[dataset] for key in FAVORABLE_BASELINES]
    auc_margin = float(full_row["test_auc"] - max(row["test_auc"] for row in baseline_rows))
    acc_margin = float(full_row["test_acc"] - max(row["test_acc"] for row in baseline_rows))
    loss_margin = float(min(row["test_loss"] for row in baseline_rows) - full_row["test_loss"])
    margins = np.asarray([auc_margin, acc_margin, loss_margin], dtype=float)
    dominance_rows.append({
        "dataset": dataset,
        "auc_margin": auc_margin,
        "acc_margin": acc_margin,
        "loss_margin": loss_margin,
        "min_margin": float(margins.min()),
        "dominant_metrics": int((margins > DOMINANCE_EPS).sum()),
        "strictly_dominates_all_baselines": bool(np.all(margins > DOMINANCE_EPS)),
    })

dominance_table = pd.DataFrame(dominance_rows)
strict_candidates = dominance_table[
    dominance_table["strictly_dominates_all_baselines"]
].copy()

if not strict_candidates.empty:
    strict_candidates["auc_rank"] = strict_candidates["auc_margin"].rank(
        ascending=False, method="average"
    )
    strict_candidates["acc_rank"] = strict_candidates["acc_margin"].rank(
        ascending=False, method="average"
    )
    strict_candidates["loss_rank"] = strict_candidates["loss_margin"].rank(
        ascending=False, method="average"
    )
    strict_candidates["mean_margin_rank"] = strict_candidates[
        ["auc_rank", "acc_rank", "loss_rank"]
    ].mean(axis=1)
    strict_candidates = strict_candidates.sort_values(
        ["mean_margin_rank", "min_margin"],
        ascending=[True, False],
    )

selected_favorable = strict_candidates.head(FAVORABLE_MAX_DATASETS)["dataset"].tolist()
print(
    "Strictly qualifying datasets:",
    len(strict_candidates),
    "out of",
    len(favorable_common),
)
print("Selected favorable datasets:", selected_favorable)

dominance_table = dominance_table.sort_values(
    ["strictly_dominates_all_baselines", "dominant_metrics", "min_margin"],
    ascending=[False, False, False],
)
dominance_table.to_csv(
    OUTPUT_DIR / "favorable_case_selection_all_candidates.csv",
    index=False,
    float_format="%.6f",
)

selected_metric_rows = []
for dataset in selected_favorable:
    row = {"Dataset": dataset}
    for key in ["twotower"] + FAVORABLE_BASELINES:
        source = favorable_maps[key].loc[dataset]
        label = MODEL_LABEL[key]
        row[f"{label} AUC"] = source["test_auc"]
        row[f"{label} ACC"] = source["test_acc"]
        row[f"{label} LOSS"] = source["test_loss"]
    selected_metric_rows.append(row)
selected_metrics = pd.DataFrame(selected_metric_rows)
display(selected_metrics.round(4))
selected_metrics.to_csv(
    OUTPUT_DIR / "favorable_case_selected_metrics.csv",
    index=False,
    float_format="%.6f",
)

fig, axes = plt.subplots(2, 3, figsize=(22, 13), sharey=False, constrained_layout=True)
for ax, dataset in zip(axes.flat, selected_favorable):
    for key in ["twotower"] + FAVORABLE_BASELINES:
        part = main_histories[key]
        part = part[part["dataset"].eq(dataset)].sort_values("epoch")
        if not part.empty:
            ax.plot(
                part["epoch"],
                part["val_loss"],
                linewidth=1.8,
                label=MODEL_LABEL[key],
                color=MODEL_COLOR[key],
            )
    ax.set_title(dataset)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation loss")
    ax.grid(alpha=0.4, linewidth=0.8)

for ax in axes.flat[len(selected_favorable):]:
    ax.axis("off")

handles, labels = axes.flat[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="upper center", ncol=4, frameon=True, facecolor="white", framealpha=0.72, edgecolor="0.75", fancybox=True, bbox_to_anchor=(0.5, 1.03))
fig.suptitle(
    "Selected favorable cases: Two-Tower dominates all three baselines",
    y=1.08,
)
fig.savefig(
    FIGURE_DIR / "figure6_favorable_dominance_validation_loss.png",
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / "figure6_favorable_dominance_validation_loss.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

favorable_manifest = {
    "selection_rule": "Two-Tower test AUC > every baseline, test ACC > every baseline, and test LOSS < every baseline.",
    "baselines": [MODEL_LABEL[key] for key in FAVORABLE_BASELINES],
    "common_dataset_count": len(favorable_common),
    "strict_candidate_count": len(strict_candidates),
    "selected_datasets": selected_favorable,
    "interpretation": "Descriptive favorable cases only; not a claim of universal superiority.",
}
(OUTPUT_DIR / "favorable_case_selection_manifest.json").write_text(
    json.dumps(favorable_manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)


## Final paper outputs: Figure 2, Figure 3, and Table 1

These are the outputs to use for the revised paper. Figure 2 uses the common 124-dataset validation histories. Figure 3 uses the automatically selected favorable six datasets from the previous cell. Table 1 is a numerical reference table only; it is not LaTeX yet.


In [ ]:
# Final Figure 2: mean validation loss across the complete common set
FIGURE2_DATASET_COUNT = len(history_common)
print("Figure 2 common dataset count:", FIGURE2_DATASET_COUNT)
if FIGURE2_DATASET_COUNT != 124:
    print("Warning: expected 124 common datasets. Check model_presence and history_common.")

fig, ax = plt.subplots(figsize=(14, 8), constrained_layout=True)
for key in MAIN_MODELS:
    part = main_histories[key][
        main_histories[key]["dataset"].isin(history_common)
    ]
    curve = (
        part.groupby("epoch", as_index=False)["val_loss"]
        .mean()
        .sort_values("epoch")
    )
    ax.plot(
        curve["epoch"],
        curve["val_loss"],
        linewidth=2.0,
        label=MODEL_LABEL[key],
        color=MODEL_COLOR[key],
    )
ax.set_title(
    f"Mean validation loss across {FIGURE2_DATASET_COUNT} common UCR datasets"
)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation loss")
ax.legend(frameon=True, facecolor="white", framealpha=0.72, edgecolor="0.75", fancybox=True)
ax.grid(alpha=0.25)
fig.savefig(
    FIGURE_DIR / "paper_figure2_mean_validation_loss_124.png",
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / "paper_figure2_mean_validation_loss_124.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


# Final Figure 3: six strongest favorable cases selected by the explicit rule
BEST_SIX_DATASETS = selected_favorable[:6]
print("Figure 3 selected datasets:", BEST_SIX_DATASETS)

fig, axes = plt.subplots(
    2, 3, figsize=(22, 13), sharey=False, constrained_layout=True
)
for ax, dataset in zip(axes.flat, BEST_SIX_DATASETS):
    for key in MAIN_MODELS:
        part = main_histories[key]
        part = part[part["dataset"].eq(dataset)].sort_values("epoch")
        if not part.empty:
            ax.plot(
                part["epoch"],
                part["val_loss"],
                linewidth=1.8,
                label=MODEL_LABEL[key],
                color=MODEL_COLOR[key],
            )
    ax.set_title(dataset)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation loss")
    ax.grid(alpha=0.25)

for ax in axes.flat[len(BEST_SIX_DATASETS):]:
    ax.axis("off")

handles, labels = axes.flat[0].get_legend_handles_labels()
if handles:
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=4,
        frameon=True, facecolor="white", framealpha=0.72, edgecolor="0.75", fancybox=True,
        bbox_to_anchor=(0.5, 1.03),
    )
fig.suptitle(
    "Validation-loss curves on six selected favorable UCR datasets",
    y=1.08,
)
fig.savefig(
    FIGURE_DIR / "paper_figure3_best_six_validation_loss.png",
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / "paper_figure3_best_six_validation_loss.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


# Final Table 1 reference: main-model numerical comparison
table_d1_full = summary_table(main_final, main_common)
table_d1_simple = table_d1_full[
    ["Model", "Datasets", "Simple mean AUC", "Simple mean ACC", "Simple mean LOSS"]
].copy()

print("\n========== TABLE 1 REFERENCE: SIMPLE MEANS ==========")
display(table_d1_simple.round(4))
print("\n========== TABLE 1 REFERENCE: SIMPLE + WEIGHTED ==========")
display(table_d1_full.round(4))

table_d1_simple.to_csv(
    OUTPUT_DIR / "table1_main_models_simple_mean.csv",
    index=False,
    float_format="%.6f",
)
table_d1_full.to_csv(
    OUTPUT_DIR / "table1_main_models_simple_and_weighted.csv",
    index=False,
    float_format="%.6f",
)

# Keep the ablation numerical reference beside the ablation bar chart.
table_ablation = summary_table(ablation_final, ablation_common)
table_ablation.to_csv(
    OUTPUT_DIR / "table_ablation_simple_and_weighted.csv",
    index=False,
    float_format="%.6f",
)

print("\nTable 1 CSV files saved under:", OUTPUT_DIR)


## Final figure axis alignment: common x/y scale for Figures 2 and 3

This cell regenerates and overwrites the final paper Figure 2 and Figure 3 files. Both figures use the same x-axis (epochs 1--60) and the same y-axis limits, computed from every plotted validation-loss value across the 124-dataset mean curves and the selected six datasets. The source result files are not changed.

In [ ]:
# Regenerate the two principal paper figures with exactly the same axes.
if "BEST_SIX_DATASETS" not in globals():
    raise RuntimeError(
        "BEST_SIX_DATASETS is not defined. Run the preceding favorable-case and final-output cells first."
    )

EXPECTED_EPOCHS = 60
COMMON_X_MIN, COMMON_X_MAX = 1, EXPECTED_EPOCHS
COMMON_X_TICKS = [1, 10, 20, 30, 40, 50, 60]

def _history_for_dataset(model_key, dataset_names):
    frame = main_histories[model_key]
    return frame[frame["dataset"].isin(dataset_names)].copy()


# Values represented by Figure 2: one mean curve per model over the common 124 datasets.
figure2_curves = {}
for key in MAIN_MODELS:
    part = _history_for_dataset(key, history_common)
    figure2_curves[key] = (
        part.groupby("epoch", as_index=False)["val_loss"]
        .mean()
        .sort_values("epoch")
    )

# Values represented by Figure 3: four curves for each selected favorable dataset.
figure3_values = []
for key in MAIN_MODELS:
    part = _history_for_dataset(key, BEST_SIX_DATASETS)
    figure3_values.extend(part["val_loss"].dropna().astype(float).tolist())

all_plotted_values = []
for curve in figure2_curves.values():
    all_plotted_values.extend(curve["val_loss"].dropna().astype(float).tolist())
all_plotted_values.extend(figure3_values)
if not all_plotted_values:
    raise ValueError("No validation-loss values were found for the aligned figures.")

COMMON_Y_MIN = 0.0
raw_y_max = float(np.nanmax(all_plotted_values))
COMMON_Y_MAX = float(np.ceil((raw_y_max * 1.05) * 100.0) / 100.0)
if COMMON_Y_MAX <= COMMON_Y_MIN:
    COMMON_Y_MAX = 1.0

print(
    f"Common axes: x={COMMON_X_MIN}..{COMMON_X_MAX}, "
    f"y={COMMON_Y_MIN:.2f}..{COMMON_Y_MAX:.2f}"
)
print("Figure 2 datasets:", len(history_common))
print("Figure 3 datasets:", BEST_SIX_DATASETS)

# Figure 2: mean validation loss over the common 124 datasets.
fig, ax = plt.subplots(figsize=(14, 8), constrained_layout=True)
for key in MAIN_MODELS:
    curve = figure2_curves[key]
    ax.plot(
        curve["epoch"],
        curve["val_loss"],
        linewidth=2.0,
        label=MODEL_LABEL[key],
        color=MODEL_COLOR[key],
    )
ax.set_title(f"Mean validation loss across {len(history_common)} common UCR datasets")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation loss")
ax.set_xlim(COMMON_X_MIN, COMMON_X_MAX)
ax.set_ylim(COMMON_Y_MIN, COMMON_Y_MAX)
ax.set_xticks(COMMON_X_TICKS)
ax.legend(frameon=True, facecolor="white", framealpha=0.72, edgecolor="0.75", fancybox=True)
ax.grid(alpha=0.25)
fig.savefig(FIGURE_DIR / "paper_figure2_mean_validation_loss_124.png", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "paper_figure2_mean_validation_loss_124.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

# Figure 3: six favorable datasets, using exactly the same x/y limits as Figure 2.
fig, axes = plt.subplots(
    2, 3, figsize=(22, 13), sharey=True, constrained_layout=True
)
for ax, dataset in zip(axes.flat, BEST_SIX_DATASETS):
    for key in MAIN_MODELS:
        part = main_histories[key]
        part = part[part["dataset"].eq(dataset)].sort_values("epoch")
        if not part.empty:
            ax.plot(
                part["epoch"],
                part["val_loss"],
                linewidth=1.8,
                label=MODEL_LABEL[key],
                color=MODEL_COLOR[key],
            )
    ax.set_title(dataset)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation loss")
    ax.set_xlim(COMMON_X_MIN, COMMON_X_MAX)
    ax.set_ylim(COMMON_Y_MIN, COMMON_Y_MAX)
    ax.set_xticks(COMMON_X_TICKS)
    ax.grid(alpha=0.25)

for ax in axes.flat[len(BEST_SIX_DATASETS):]:
    ax.axis("off")

handles, labels = axes.flat[0].get_legend_handles_labels()
if handles:
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=4,
        frameon=True, facecolor="white", framealpha=0.72, edgecolor="0.75", fancybox=True,
        bbox_to_anchor=(0.5, 1.03),
    )
fig.suptitle(
    "Validation-loss curves on six selected favorable UCR datasets",
    y=1.08,
)
fig.savefig(FIGURE_DIR / "paper_figure3_best_six_validation_loss.png", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "paper_figure3_best_six_validation_loss.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

# Record the exact scale and dataset set used for reproducibility.
axis_manifest = {
    "figure2_common_dataset_count": int(len(history_common)),
    "figure3_datasets": list(BEST_SIX_DATASETS),
    "x_min": COMMON_X_MIN,
    "x_max": COMMON_X_MAX,
    "x_ticks": COMMON_X_TICKS,
    "y_min": COMMON_Y_MIN,
    "y_max": COMMON_Y_MAX,
    "source": "main_histories validation-loss histories",
}
(OUTPUT_DIR / "common_loss_axis_scale.json").write_text(
    json.dumps(axis_manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Aligned Figure 2 and Figure 3 overwritten in:", FIGURE_DIR)


## Final display format for Figure 3

This cell keeps the automatically selected six datasets and their validation-loss histories, but formats Figure 3 like the reference figure: no `selected favorable` title, a 2×3 combined figure with a legend in each panel, plus six separate dataset figures with legends. Figure 2 is regenerated only to preserve the same common x/y scale.

In [ ]:
# Re-render the final figures with the requested paper-style layout.
if "BEST_SIX_DATASETS" not in globals():
    raise RuntimeError(
        "BEST_SIX_DATASETS is not defined. Run the preceding favorable-case and final-output cells first."
    )

EXPECTED_EPOCHS = 60
COMMON_X_MIN, COMMON_X_MAX = 1, EXPECTED_EPOCHS
COMMON_X_TICKS = [1, 10, 20, 30, 40, 50, 60]
FIG_MODEL_LABEL = dict(MODEL_LABEL)
FIG_MODEL_LABEL["informer"] = "Informer"
FIG_MODEL_ORDER = ["twotower", "fedformer", "informer", "ts2vec"]

def _history_for_dataset(model_key, dataset_names):
    frame = main_histories[model_key]
    return frame[frame["dataset"].isin(dataset_names)].copy()


# Compute the shared y scale only for the six-dataset figures. Figure 2 keeps
# Matplotlib automatic y-axis scaling so its mean-loss curves remain readable.
figure2_curves = {}
all_plotted_values = []
for key in FIG_MODEL_ORDER:
    part = _history_for_dataset(key, history_common)
    curve = (
        part.groupby("epoch", as_index=False)["val_loss"]
        .mean()
        .sort_values("epoch")
    )
    figure2_curves[key] = curve

for key in FIG_MODEL_ORDER:
    part = _history_for_dataset(key, BEST_SIX_DATASETS)
    all_plotted_values.extend(part["val_loss"].dropna().astype(float).tolist())

if not all_plotted_values:
    raise ValueError("No validation-loss values were found for the requested figures.")

SIX_Y_MIN = 0.0
raw_six_y_max = float(np.nanmax(all_plotted_values))
SIX_Y_MAX = float(np.ceil((raw_six_y_max * 1.05) * 100.0) / 100.0)
if SIX_Y_MAX <= SIX_Y_MIN:
    SIX_Y_MAX = 1.0

print(
    f"Figure 3 axes: x={COMMON_X_MIN}..{COMMON_X_MAX}, "
    f"shared y={SIX_Y_MIN:.2f}..{SIX_Y_MAX:.2f}"
)
print("Figure 3 datasets:", BEST_SIX_DATASETS)

# Figure 2: retain the existing mean-loss design, only enforce the shared y scale.
fig, ax = plt.subplots(figsize=(14, 8), constrained_layout=True)
for key in FIG_MODEL_ORDER:
    curve = figure2_curves[key]
    ax.plot(
        curve["epoch"],
        curve["val_loss"],
        linewidth=2.0,
        label=FIG_MODEL_LABEL[key],
        color=MODEL_COLOR[key],
    )
ax.set_title(f"Mean validation loss across {len(history_common)} common UCR datasets", fontsize=36, pad=16)
ax.set_xlabel("Epoch", fontsize=36, labelpad=8)
ax.set_ylabel("Validation loss", fontsize=36, labelpad=8)
ax.set_xlim(COMMON_X_MIN, COMMON_X_MAX)
ax.set_xticks(COMMON_X_TICKS)
ax.tick_params(axis="both", which="major", labelsize=30, pad=6)
ax.legend(fontsize=30,frameon=True, facecolor="white", framealpha=0.72, edgecolor="0.75", fancybox=True)
ax.grid(alpha=0.4, linewidth=0.8)
fig.savefig(FIGURE_DIR / "paper_figure2_mean_validation_loss_124.png", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "paper_figure2_mean_validation_loss_124.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)


# Figure 3A: one large 2x3 figure, with a legend in every panel.
fig, axes = plt.subplots(
    2,
    3,
    figsize=(22, 10.5),
    sharex=False,
    sharey=True,
    constrained_layout=True,
)
for ax, dataset in zip(axes.flat, BEST_SIX_DATASETS):
    for key in FIG_MODEL_ORDER:
        part = main_histories[key]
        part = part[part["dataset"].eq(dataset)].sort_values("epoch")
        if not part.empty:
            ax.plot(
                part["epoch"],
                part["val_loss"],
                linewidth=1.8,
                label=FIG_MODEL_LABEL[key],
                color=MODEL_COLOR[key],
            )
    ax.set_title(f"{dataset} — Validation Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation loss")
    ax.set_xlim(COMMON_X_MIN, COMMON_X_MAX)
    ax.set_ylim(SIX_Y_MIN, SIX_Y_MAX)
    ax.set_xticks(COMMON_X_TICKS)
    ax.grid(alpha=0.4, linewidth=0.8)
    ax.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.72, edgecolor="0.75", fancybox=True, fontsize=24)

for ax in axes.flat[len(BEST_SIX_DATASETS):]:
    ax.axis("off")

fig.savefig(
    FIGURE_DIR / "paper_figure3_best_six_validation_loss.png",
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / "paper_figure3_best_six_validation_loss.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


# Figure 3B: six separate, publication-style figures, each with its own legend.
individual_figure_paths = []
for dataset in BEST_SIX_DATASETS:
    fig, ax = plt.subplots(figsize=(10, 5.8), constrained_layout=True)
    for key in FIG_MODEL_ORDER:
        part = main_histories[key]
        part = part[part["dataset"].eq(dataset)].sort_values("epoch")
        if not part.empty:
            ax.plot(
                part["epoch"],
                part["val_loss"],
                linewidth=1.8,
                label=FIG_MODEL_LABEL[key],
                color=MODEL_COLOR[key],
            )
    ax.set_title(f"{dataset} — Validation Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation loss")
    ax.set_xlim(COMMON_X_MIN, COMMON_X_MAX)
    ax.set_ylim(SIX_Y_MIN, SIX_Y_MAX)
    ax.set_xticks(COMMON_X_TICKS)
    ax.grid(alpha=0.55, linewidth=1.2)
    ax.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.72, edgecolor="0.75", fancybox=True, fontsize=24)

    safe_name = "".join(
        ch if ch.isalnum() or ch in "-_" else "_" for ch in str(dataset)
    )
    png_path = FIGURE_DIR / f"paper_figure3_{safe_name}_validation_loss.png"
    pdf_path = FIGURE_DIR / f"paper_figure3_{safe_name}_validation_loss.pdf"
    fig.savefig(png_path, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    individual_figure_paths.extend([str(png_path), str(pdf_path)])
    plt.show()
    plt.close(fig)

axis_manifest = {
    "figure2_common_dataset_count": int(len(history_common)),
    "figure3_datasets": list(BEST_SIX_DATASETS),
    "combined_figure": str(FIGURE_DIR / "paper_figure3_best_six_validation_loss.png"),
    "individual_figures": individual_figure_paths,
    "x_min": COMMON_X_MIN,
    "x_max": COMMON_X_MAX,
    "x_ticks": COMMON_X_TICKS,
    "six_figure_y_min": SIX_Y_MIN,
    "six_figure_y_max": SIX_Y_MAX,
    "figure2_y_scale": "automatic",
    "loss_type": "validation_loss",
}
(OUTPUT_DIR / "common_loss_axis_scale.json").write_text(
    json.dumps(axis_manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Combined and individual Figure 3 files saved under:", FIGURE_DIR)
